# SRGAN training: reconstruction pretraining then adversarial training
Use only persisted training examples. First pretrain the generator with L1 reconstruction loss. Then begin adversarial generator/discriminator training. Both stages resume automatically and save checkpoints plus bicubic/generated/real sample grids every five epochs.

In [ ]:
from dataclasses import asdict
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from torch.utils.data import DataLoader

from applied_ai_midterm.config import load_config
from applied_ai_midterm.data import SRGANPathDataset, load_splits
from applied_ai_midterm.models import Discriminator, Generator, GeneratorLoss, VGGPerceptualFeatures
from applied_ai_midterm.training import (
    pretrain_generator,
    resume_generator_pretraining,
    resume_latest_srgan,
    seed_everything,
    select_device,
    train_srgan,
)
from applied_ai_midterm.transforms import SRGANPairTransform, denormalize_srgan

In [ ]:
config = load_config(Path("configs/config.yaml"))
assert config.srgan_epochs >= 150
assert config.checkpoint_interval == 5
seed_everything(config.random_seed)
device = select_device()
splits = load_splits(Path("data/splits"))
srgan_dataset = SRGANPathDataset(
    splits.train,  # Reserved test examples never enter either SRGAN stage.
    SRGANPairTransform(
        training=True,
        low_resolution_size=config.low_resolution_size,
        high_resolution_size=config.high_resolution_size,
    ),
)
loader_generator = torch.Generator().manual_seed(config.random_seed)
srgan_loader = DataLoader(
    srgan_dataset,
    batch_size=config.srgan_batch_size,
    shuffle=True,
    num_workers=config.num_workers,
    generator=loader_generator,
)
fixed_low_resolution, fixed_high_resolution = next(iter(srgan_loader))
sample_tensors = (fixed_low_resolution[:4], fixed_high_resolution[:4])
sample_dir = Path("artifacts/srgan_samples")
print(f"Training examples: {len(srgan_dataset)}; device: {device}")

## Stage 1 — generator reconstruction pretraining
This stage uses only L1 pixel/content reconstruction loss. The default below is 20 epochs and can resume from the latest `generator_pretrain_epoch_*.pt` file.

In [ ]:
generator = Generator().to(device)
pretrain_epochs = 20
pretrain_checkpoint_dir = Path("checkpoints/srgan_pretrain")
pretrain_optimizer = torch.optim.Adam(generator.parameters(), lr=1e-4, betas=(0.9, 0.999))
pretrain_scheduler = torch.optim.lr_scheduler.StepLR(pretrain_optimizer, step_size=10, gamma=0.5)
pretrain_start_epoch, reconstruction_history, pretrain_resume_path = resume_generator_pretraining(
    pretrain_checkpoint_dir,
    generator=generator,
    optimizer=pretrain_optimizer,
    scheduler=pretrain_scheduler,
    device=device,
)
print(f"Resuming pretraining from {pretrain_resume_path}" if pretrain_resume_path else "Starting generator pretraining")
reconstruction_history = pretrain_generator(
    generator,
    srgan_loader,
    pretrain_optimizer,
    pretrain_checkpoint_dir,
    total_epochs=pretrain_epochs,
    start_epoch=pretrain_start_epoch,
    device=device,
    config=asdict(config),
    history=reconstruction_history,
    checkpoint_interval=config.checkpoint_interval,
    scheduler=pretrain_scheduler,
    sample_tensors=sample_tensors,
    sample_dir=sample_dir,
    sample_interval=5,
)

## Stage 2 — adversarial generator/discriminator training
The pretrained generator now uses pixel/content, adversarial, and optional VGG perceptual losses. Complete adversarial checkpoints resume both networks, both optimizers, both schedulers, and history.

In [ ]:
discriminator = Discriminator().to(device)
generator_optimizer = torch.optim.Adam(generator.parameters(), lr=1e-4, betas=(0.9, 0.999))
discriminator_optimizer = torch.optim.Adam(discriminator.parameters(), lr=1e-4, betas=(0.9, 0.999))
generator_scheduler = torch.optim.lr_scheduler.StepLR(generator_optimizer, step_size=50, gamma=0.5)
discriminator_scheduler = torch.optim.lr_scheduler.StepLR(discriminator_optimizer, step_size=50, gamma=0.5)
use_vgg_perceptual_loss = True
perceptual_features = VGGPerceptualFeatures().to(device) if use_vgg_perceptual_loss else None
generator_loss = GeneratorLoss(perceptual_features=perceptual_features).to(device)
adversarial_checkpoint_dir = Path("checkpoints/srgan")
start_epoch, history, adversarial_resume_path = resume_latest_srgan(
    adversarial_checkpoint_dir,
    generator=generator,
    discriminator=discriminator,
    generator_optimizer=generator_optimizer,
    discriminator_optimizer=discriminator_optimizer,
    generator_scheduler=generator_scheduler,
    discriminator_scheduler=discriminator_scheduler,
    device=device,
)
print(f"Resuming adversarial training from {adversarial_resume_path}" if adversarial_resume_path else "Starting adversarial training from pretrained generator")
history = train_srgan(
    generator,
    discriminator,
    srgan_loader,
    generator_optimizer,
    discriminator_optimizer,
    generator_loss,
    adversarial_checkpoint_dir,
    total_epochs=config.srgan_epochs,
    start_epoch=start_epoch,
    device=device,
    config=asdict(config),
    history=history,
    checkpoint_interval=config.checkpoint_interval,
    generator_scheduler=generator_scheduler,
    discriminator_scheduler=discriminator_scheduler,
    sample_tensors=sample_tensors,
    sample_dir=sample_dir,
    sample_interval=5,
)

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(reconstruction_history)
axes[0].set(title="Generator pretraining", xlabel="Epoch", ylabel="L1 reconstruction loss")
pd.DataFrame(history).plot(ax=axes[1])
axes[1].set(title="Adversarial SRGAN training", xlabel="Epoch", ylabel="Loss")
figure.tight_layout()

In [ ]:
generator.eval()
with torch.inference_mode():
    generated = generator(fixed_low_resolution[:1].to(device)).cpu()[0]
figure, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(denormalize_srgan(fixed_low_resolution[0]).permute(1, 2, 0))
axes[0].set_title("32×32 input")
axes[1].imshow(denormalize_srgan(generated).permute(1, 2, 0))
axes[1].set_title("Generated 128×128")
axes[2].imshow(denormalize_srgan(fixed_high_resolution[0]).permute(1, 2, 0))
axes[2].set_title("Real 128×128 target")
for axis in axes:
    axis.axis("off")
figure.tight_layout()